# Device Management

GPU workflows come with one persistent chore: figuring out which device each model lives on, handling the case where that device is busy, and cleaning up afterwards when you are done. `DeviceManager` handles this. You set the `device` field on a tool's config, and DeviceManager tracks every allocation, places models on free GPUs, and evicts the least-recently-used worker when memory runs out.

The defaults cover most workloads without configuration: one model per GPU, least-recently-used eviction, full restart on reuse. This guide covers what DeviceManager does by default and the configuration available when the defaults don't fit your setup.

In [ ]:
from proto_tools.tools.structure_prediction.esmfold import (
    run_esmfold, ESMFoldInput, ESMFoldConfig,
)
from proto_tools.utils.tool_instance import ToolInstance
from proto_tools.utils.device_manager import DeviceManager, OffloadStrategy

---

## 1. Requesting a Device

Every tool's config has a `device` field. GPU tools default to `"cuda"` and CPU tools default to `"cpu"`, but you can request something more specific. The strings you pass fall into two categories, which control how much say DeviceManager has in where the model lands.

### General requests (DeviceManager chooses the GPU)

Ask for a class of device and let DeviceManager pick. This is what you want most of the time; you don't need to know which GPUs happen to be free, and if everything is busy, DeviceManager handles eviction for you.

| Value | Meaning |
|---|---|
| `"cpu"` | Run on CPU |
| `"cuda"` | Run on one GPU, DeviceManager picks which one |
| `"cudax2"`, `"cudax3"`, ... | Run on N GPUs, DeviceManager picks which ones |

### Specific requests (you choose the GPU)

Name an exact device. DeviceManager honors the request and evicts whatever is currently there if the slot is taken. Reach for this when you have a reason to pin to a known device: benchmarking, affinity with other work on the same card, or reproducing a specific placement.

| Value | Meaning |
|---|---|
| `"cuda:0"` | Run on exactly GPU 0 |
| `"cuda:0,1"` or `"cuda:0,cuda:1"` | Run on exactly GPUs 0 and 1 |

### Example: General vs Specific Allocation

To make the difference concrete, here are two calls with identical inputs but different device requests. The first asks for any GPU; the second pins to device 2.

In [ ]:
# General request: DeviceManager picks the GPU
with ToolInstance.persist():
    output = run_esmfold(ESMFoldInput(complexes=["MKTLLILAVVAAALA"]))

# Specific request: we pick the GPU
with ToolInstance.persist():
    output = run_esmfold(
        ESMFoldInput(complexes=["MKTLLILAVVAAALA"]),
        ESMFoldConfig(device="cuda:2"),
    )

<img src="assets/device-management/general-specific.svg" alt="General request lands on any free GPU; specific request pins to the requested device" width="560">

### Limiting Managed Devices

By default, DeviceManager treats every visible GPU as part of its allocation pool. On a shared machine, or when you are deliberately reserving some cards for another workload, you can narrow the pool to a subset. Either set the `BIO_TOOLS_MANAGED_DEVICES` environment variable before any tool runs, or call `configure(managed_devices=...)` at runtime.

In [ ]:
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:1"])

# General requests now only land on cuda:1
with ToolInstance.persist():
    output = run_esmfold(ESMFoldInput(complexes=["MKTLLILAVVAAALA"]))

# Equivalent via environment variable:
# export BIO_TOOLS_MANAGED_DEVICES="cuda:1"

# Reset back to full pool
DeviceManager.reset_instance()

<img src="assets/device-management/managed.svg" alt="Managed pool restricts general requests to the listed GPUs; the rest are reserved for other work" width="560">

General requests now only land on managed GPUs. Unmanaged cards stay completely untouched by DeviceManager and are free for other work on the same machine.

---

## 2. Eviction Strategies

DeviceManager's allocation pool is finite. When every managed GPU is full and a new model needs to load, something has to give. DeviceManager evicts the **least recently used** worker to make room, but what "evict" actually means is up to you. The choice is controlled by `offload_strategy` (or the `BIO_TOOLS_OFFLOAD_STRATEGY` environment variable).

- **RESTART** (default) shuts the evicted worker down entirely. Frees all memory, but the next call to that tool pays the full startup cost to reload.
- **CPU** moves the evicted model into system RAM. It stays loaded, and returning it to GPU later is a fast copy rather than a full reload.

Use RESTART when GPU memory is your tightest constraint and you are fine paying the reload cost on an occasional wake-up. Use CPU when you have RAM to spare and you cycle between a small handful of models frequently enough that cold reloads hurt.

<Note>
Not every tool supports staying resident on GPU between calls, and not every tool can be offloaded to CPU. For models that don't, DeviceManager falls back to RESTART behavior regardless of the strategy you set, and every call pays the full startup cost.
</Note>

### RESTART (Default)

The default strategy. When eviction is needed, DeviceManager terminates the evicted worker's subprocess. The GPU slot is fully freed and the new model moves in immediately.

In [ ]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0"])

with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=["MKTLLILAVVAAALA"]), instance="A")
    # A: cuda:0, ~15 GB used

    # other tasks here ...

    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        run_esmfold(ESMFoldInput(complexes=["GAVLTVLLGGLLLA"]), instance="B")
        # RESTART evicts A; A fully shut down
        # B: cuda:0, ~15 GB used

DeviceManager.reset_instance()

<img src="assets/device-management/restart.svg" alt="RESTART eviction: A is fully shut down when B needs cuda:0" width="560">

A later call to the evicted tool behaves exactly like its first call ever did: fresh subprocess, fresh model load, full cold-start cost.

### CPU Offload

With `offload_strategy=OffloadStrategy.CPU`, an evicted worker is not torn down. DeviceManager moves its weights onto CPU memory and keeps the process alive. Promoting it back to GPU later is just a tensor copy, which is orders of magnitude faster than a fresh model load.

In [ ]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0"], offload_strategy=OffloadStrategy.CPU)

with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=["MKTLLILAVVAAALA"]), instance="A")
    # A: cuda:0

    # other tasks here ...

    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        run_esmfold(ESMFoldInput(complexes=["GAVLTVLLGGLLLA"]), instance="B")
        # A moved to CPU, B: cuda:0

        # other tasks here ...

        run_esmfold(ESMFoldInput(complexes=["MKTLLILAVVAAALA"]), instance="A")
        # Fast CPU → GPU swap (~7s vs ~15s cold reload)
        # A: cuda:0, B: cpu

DeviceManager.reset_instance()

<img src="assets/device-management/cpu-offload.svg" alt="CPU offload: evicted models move to CPU memory and come back fast when re-activated" width="560">

The trade is RAM. Every offloaded worker is still resident, just not on the GPU. If you have more models than will fit across both GPU and system memory, stick with RESTART.

### LRU Eviction Across Multiple GPUs

LRU is simple in spirit: every call to `run_*` updates the last-used time on its worker. When a new allocation needs a GPU and every GPU is occupied, DeviceManager walks its allocation map and picks whichever worker's last-used time is oldest, regardless of which GPU it sits on.

In [ ]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0", "cuda:1"])

with ToolInstance.persist_tool("esmfold", instance_name="A"):
    run_esmfold(ESMFoldInput(complexes=["MKTLLILAVVAAALA"]), instance="A")
    # A: cuda:0

    # other tasks here ...

    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        run_esmfold(ESMFoldInput(complexes=["GAVLTVLLGGLLLA"]), instance="B")
        # A: cuda:0, B: cuda:1

        # other tasks here ...

        with ToolInstance.persist_tool("esmfold", instance_name="C"):
            run_esmfold(ESMFoldInput(complexes=["MGQQPGKVLGDQRR"]), instance="C")
            # A is the least recently used; evicted from cuda:0
            # B: cuda:1, C: cuda:0

DeviceManager.reset_instance()

<img src="assets/device-management/lru.svg" alt="LRU eviction with 2 GPUs and 3 instances: A is evicted because it was used least recently" width="560">

The specific GPU the LRU worker happens to be on doesn't matter; DeviceManager picks the oldest worker anywhere in the pool and reuses its slot.

---

## 3. Multiple Models Per Device

DeviceManager's default policy is one worker per GPU. If you have memory headroom to spare, say a GPU with 80 GB and two 15 GB models that both need to be live, you can pack multiple workers onto the same card rather than making them fight over it.

<Note>
DeviceManager does not currently track model sizes or estimate memory usage. It is on you to make sure the models you are packing actually fit. If you overcommit, you will get an OOM error.
</Note>

### Packing 4 Instances on 2 GPUs

Enable `allow_multiple_per_device=True` and new allocations round-robin across the pool instead of triggering eviction. Four instances across two GPUs land two on each.

In [ ]:
DeviceManager.reset_instance()
dm = DeviceManager.get_instance()
dm.configure(managed_devices=["cuda:0", "cuda:1"], allow_multiple_per_device=True)

with ToolInstance.persist_tool("esmfold", instance_name="A"):
    with ToolInstance.persist_tool("esmfold", instance_name="B"):
        with ToolInstance.persist_tool("esmfold", instance_name="C"):
            with ToolInstance.persist_tool("esmfold", instance_name="D"):

                run_esmfold(ESMFoldInput(complexes=["MKTLLILAVVAAALA"]), instance="A")
                run_esmfold(ESMFoldInput(complexes=["GAVLTVLLGGLLLA"]), instance="B")
                run_esmfold(ESMFoldInput(complexes=["MGQQPGKVLGDQRR"]), instance="C")
                run_esmfold(ESMFoldInput(complexes=["ASTVKFLGPVLDAA"]), instance="D")
                # A: cuda:0, B: cuda:1, C: cuda:0, D: cuda:1

DeviceManager.reset_instance()

<img src="assets/device-management/packing.svg" alt="Four instances packed across two GPUs: A, C on cuda:0 and B, D on cuda:1" width="560">

Inside the nested `with` blocks every instance stays loaded. No eviction, no offload, no restarts. The ceiling is GPU memory and whatever fragmentation tolerance PyTorch can manage.

---

## Configuration Reference

DeviceManager can be configured programmatically or via environment variables.

### Environment Variables

| Variable | Example | Description |
|---|---|---|
| `BIO_TOOLS_MANAGED_DEVICES` | `"cuda:0,cuda:1"` | Restrict device pool |
| `BIO_TOOLS_OFFLOAD_STRATEGY` | `"restart"` or `"cpu"` | Eviction strategy |
| `BIO_TOOLS_ALLOW_MULTI_DEVICE` | `"true"` or `"false"` | Multiple models per GPU |

### Programmatic Configuration

In [ ]:
from proto_tools.utils.device_manager import DeviceManager, OffloadStrategy

dm = DeviceManager.get_instance()
dm.configure(
    managed_devices=["cuda:0", "cuda:1"],
    offload_strategy=OffloadStrategy.CPU,
    allow_multiple_per_device=False,
)

## Next Steps

<CardGroup cols={2}>
  <Card title="Tool Persistence" icon="box" href="/tools/guides/tool-persistence">
    Keep models loaded across calls to avoid repeated startup cost.
  </Card>
  <Card title="Parallel Execution" icon="layers" href="/tools/guides/parallel-execution">
    Fan work out across every managed GPU with ToolPool.
  </Card>
</CardGroup>